In [ ]:
%run /Users/calvin.waldheim@gmail.com/lakebase_config

In [0]:
# Cell 1
%pip install psycopg2-binary

In [0]:
# Cell 2 - Agent
import psycopg2
import json
from mlflow.deployments import get_deploy_client


client = get_deploy_client("databricks")

def embed(text):
    response = client.predict(
        endpoint="databricks-gte-large-en",
        inputs={"input": [text]}
    )
    return response["data"][0]["embedding"]

def retrieve(query, project_id="memory-kb-poc", top_k=3):
    query_embedding = embed(query)
    conn = psycopg2.connect(CONN_STRING, password=TOKEN)
    cur = conn.cursor()
    cur.execute("""
        SELECT context, embedding <=> %s::vector AS distance
        FROM memories
        WHERE project_id = %s
        ORDER BY distance ASC
        LIMIT %s
    """, (json.dumps(query_embedding), project_id, top_k))
    results = cur.fetchall()
    cur.close()
    conn.close()
    return results

def ask(question):
    # 1. Retrieve relevant memory
    memories = retrieve(question)
    context = "\n\n".join([m[0] for m in memories])
    
    # 2. Call LLM with memory as context
    response = client.predict(
        endpoint="databricks-meta-llama-3-3-70b-instruct",
        inputs={
            "messages": [
                {"role": "system", "content": f"""You are a helpful assistant. 
Answer questions using the context below. If the context doesn't contain the answer, say so.

CONTEXT:
{context}"""},
                {"role": "user", "content": question}
            ]
        }
    )
    
    answer = response["choices"][0]["message"]["content"]
    
    # 3. Write episodic memory of this interaction
    content = f"Q: {question}\nA: {answer}"
    content_hash = __import__('hashlib').md5(content.encode()).hexdigest()
    embedding = embed(content)
    
    conn = psycopg2.connect(CONN_STRING, password=TOKEN)
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO memories 
            (project_id, project_type, memory_type, scope, domain, rule, context, source_ref, content_hash, embedding, quality_score)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING
    """, (
        "memory-kb-poc",
        "product",
        "episodic",
        "organizational",
        "interactions",
        question[:100],
        content,
        "agent-interaction",
        content_hash,
        json.dumps(embedding),
        0.9
    ))
    conn.commit()
    cur.close()
    conn.close()
    
    return answer

In [0]:
# Cell 3 - Test it
answer = ask("What did we discuss about end to end implementation?")
print(answer)

In [0]:
for endpoint in client.list_endpoints():
    print(endpoint["name"])

In [0]:
# Check what's in memory
conn = psycopg2.connect(CONN_STRING, password=TOKEN)
cur = conn.cursor()
cur.execute("""
    SELECT memory_type, domain, rule, created_at 
    FROM memories 
    WHERE project_id = 'memory-kb-poc'
    ORDER BY created_at DESC
    LIMIT 10
""")
for row in cur.fetchall():
    print(f"[{row[0]}] [{row[1]}] {row[2][:100]}")
    print(f"  created: {row[3]}\n")
cur.close()
conn.close()

In [0]:
memories = retrieve("What did we discuss about end to end implementation?")
for i, (context, distance) in enumerate(memories):
    print(f"\n--- Retrieved {i+1} (distance: {distance:.3f}) ---")
    print(context[:200])